In [ ]:
import numpy as np
from math import exp,log,prod
import matplotlib.pyplot as plt


## divide data 60-20-20

In [ ]:
with open('dane.data', 'r') as data, open('training.data', 'w') as training, open('validation.data', 'w') as validation, open('testing.data', 'w') as testing:
    for line in data:
        random_value = np.random.rand()
        if random_value < 0.6:  # 60% chance to go to training
            training.write(line)
        elif random_value < 0.8:  # 20% chance to go to validation
            validation.write(line)
        else:
            testing.write(line)

## divide 50-30-20

In [ ]:
with open('dane.data', 'r') as data, open('training.data', 'w') as training, open('validation.data', 'w') as validation, open('testing.data', 'w') as testing:
    for line in data:
        random_value = np.random.rand()
        if random_value < 0.5:  
            training.write(line)
        elif random_value < 0.8:  
            validation.write(line)
        else:
            testing.write(line)

## read_data

In [ ]:
def load_data(filename):
    data = np.loadtxt(filename, delimiter='\t')
    X = data[:, :-1]
    y = data[:, -1]
    return X, y


## scaling

In [ ]:
# MIN-MAX SCALING
def scale(X):
    X_scaled = (X - min_val) / (max_val - min_val)
    return X_scaled

In [ ]:
# MIN-MAX 2 SCALING
def scale(X):
    X_scaled = (X - mean) / (max_val - min_val)
    return X_scaled

In [ ]:
# STANDARIZATION

def scale(X):
    return (X-mean)/std

## different loss functions

In [ ]:
# square
def loss_function(a,y):
    return (a-y)**2

In [ ]:
# absolute
def loss_function(a,y):
    return abs(a-y)

## base functions

In [ ]:
#unified basis functions
def basis_identity(x):
    return list(x)

def basis_funky(x):
    return  [x[0]*x[4]]

def basis_poly(x, degree):
    result = list(x)
    result += [val**degree for val in x]
    return result

def basis_gauss(x, s):
    x_mean = sum(x) / len(x)
    return [-exp((val - x_mean)**2 / s**2) for val in x]

def basis_gauss_zero(x, s):
    return [exp(-val**2 / s**2) for val in x]

def basis_sincos(x):
    return [np.sin(val) for val in x] + [np.cos(val) for val in x]

## Regression

In [ ]:
def analytic_solution(X: np.ndarray, y: np.ndarray) -> np.ndarray:
    X = np.c_[np.ones(X.shape[0]), X]
    res = np.linalg.inv(X.T @ X) @ X.T @ y
    return res

In [ ]:
def analytic_solution_ridge(X: np.ndarray, y: np.ndarray, lam=10) -> np.ndarray:
    X = np.c_[np.ones(X.shape[0]), X]
    d = X.shape[1]
    I = np.eye(d)
    I[0, 0] = 0
    res = np.linalg.inv(X.T @ X + lam * I) @ X.T @ y
    return res

## Gradient

In [ ]:
# modify here for different learning rate
eta = 0.05

In [ ]:
# gradient descent for ridge regression
def compute_theta(theta,X,y,lamb):
    m=len(y)
    grad = X.T @ (X @ theta - y)/m
    grad[1:] +=lamb * theta[1:]
    return eta*grad

In [ ]:
# gradient descent for lasso regression
def compute_theta(theta,X,y,lamb):
    m=len(y)
    grad = X.T @(X @ theta - y)/m
    grad[1:]+= lamb*np.sign(theta[1:])
    return eta * grad

In [ ]:
# gradient descent for elastic net regression
def compute_theta(theta,X,y,lamb):
    m=len(y)
    grad = X.T @(X @ theta - y)/m
    
    grad[1:]+= lamb*np.sign(theta[1:])
    grad[1:] +=lamb * theta[1:]
    return eta * grad


In [ ]:
def gradient_descent(X, y, lam, num_iterations=1000):
    X = np.c_[np.ones(X.shape[0]), X]
    theta = np.random.random(X.shape[1]) 
    for _ in range(num_iterations):
        theta -= compute_theta(theta,X,y,lam)
    return theta

## configs:

In [ ]:
possible_configs = [
    {"type": "identity"},
    {"type": "poly", "degree": 2},
    {"type": "poly", "degree": 3},
    {"type": "poly", "degree": 4},
    {"type": "poly", "degree": 5},
    {"type": "gauss", "s": 0.5},
    {"type": "gauss", "s": 1},
    {"type": "gauss", "s": 2},
    {"type": "gauss", "s": 5},
    {"type": "gauss_zero", "s": 0.5},
    {"type": "gauss_zero", "s": 1},
    {"type": "gauss_zero", "s": 2},
    {"type": "sincos"},
    {"type": "funky"}
]

possible_lambdas = [0.01, 0.1, 1, 10, 20, 40,75, 100, 200, 400 ,500]

def remodel(X, config):
    t = config["type"]
    if t == "identity":
        return np.array([basis_identity(x) for x in X])
    elif t == "poly":
        return np.array([basis_poly(x, config["degree"]) for x in X])
    elif t == "gauss":
        return np.array([basis_gauss(x, config["s"]) for x in X])
    elif t == "gauss_zero":
        return np.array([basis_gauss_zero(x, config["s"]) for x in X])
    elif t == "sincos":
        return np.array([basis_sincos(x) for x in X])
    elif t == "funky":
        return np.array([basis_funky(x) for x in X])

## validate with gradient

In [ ]:
def validate(X_train_frac, y_train_frac):
    smallest_error = float("inf")
    best_solution  = None
    best_config    = None
    best_lambda    = None

    for config in possible_configs:
        X_train_r = remodel(X_train_frac, config)
        X_val_r = remodel(X_val, config)

        for lam in possible_lambdas:
            solution = gradient_descent(X_train_r, y_train_frac, lam=lam)
            X_val_bias = np.c_[np.ones(len(X_val_r)), X_val_r]
            error = np.mean(loss_function(X_val_bias @ solution, y_val))

            if error < smallest_error:
                smallest_error = error
                best_solution  = solution
                best_config    = config
                best_lambda    = lam

    return best_solution, best_config, best_lambda

## validate with analytic ridge

In [ ]:
def validate(X_train_frac, y_train_frac):
    smallest_error = float("inf")
    best_solution  = None
    best_config    = None
    best_lambda    = None

    for config in possible_configs:
        X_train_r = remodel(X_train_frac, config)
        X_val_r = remodel(X_val, config)

        for lam in possible_lambdas:
            solution = analytic_solution_ridge(X_train_r, y_train_frac, lam=lam)
            X_val_bias = np.c_[np.ones(len(X_val_r)), X_val_r]
            error = np.mean(loss_function(X_val_bias @ solution, y_val))

            if error < smallest_error:
                smallest_error = error
                best_solution  = solution
                best_config    = config
                best_lambda    = lam

    return best_solution, best_config, best_lambda

## TEST_FOUND_DATA

In [ ]:
def test(X_train_frac,y_train_frac):
    best_solution ,best_config, best_lambda=validate(X_train_frac,y_train_frac)
    print(f"{best_config}; lambda = {best_lambda}")
    error = 0
    X_test_r = remodel(X_test, best_config)
    for i in range(X_test_r.shape[0]):
        x = np.insert(X_test_r[i], 0, 1)

        try:
            a = best_solution @ x
            error += loss_function(a, y_test[i])
        except Exception as e:
            print(f"Błąd mnożenia: {e}")
            print(f"Kształt best_solution: {best_solution.shape}")
            print(f"Kształt x: {x.shape}")
    error /= X_test_r.shape[0]
    return error

## Random run summarized

In [ ]:
iters=20
fractions=[
    0.01,0.02,0.03,0.125,0.625,1]
res_errors=[0 for _ in range(6)]
for i in range(iters):
    with open('dane.data', 'r') as data, open('training.data', 'w') as training, open('validation.data', 'w') as validation, open('testing.data', 'w') as testing:
        for line in data:
            random_value = np.random.rand()
            if random_value < 0.6:  # 60% chance to go to training
                training.write(line)
            elif random_value < 0.8:  # 20% chance to go to validation
                validation.write(line)
            else:
                testing.write(line)

    X_train, y_train = load_data('training.data')
    X_val, y_val = load_data('validation.data')
    X_test, y_test = load_data('testing.data')
    mean=X_train.mean(axis=0)
    std = X_train.std(axis=0)
    min_val = X_train.min(axis=0)
    max_val = X_train.max(axis=0)
    X_train=scale(X_train)
    X_val=scale(X_val)
    X_test=scale(X_test)
    X_b = np.c_[np.ones(X_train.shape[0]), X_train]
    for j in range(len(fractions)):
        #  at least 1 line
        n = max(1, int(len(X_train) * fractions[j]))
        indices = np.random.choice(len(X_train), size=n, replace=False)
        X = X_train[indices]
        y = y_train[indices]
        res_errors[j]+=(test(X, y))
res_errors=[val/iters for val in res_errors]
plt.plot(fractions, res_errors, marker='o')
plt.xlabel("fraction")
plt.ylabel("error")
plt.title("Uśrednione 20 przebiegów")
plt.ylim(min(res_errors)*0.95, max(res_errors)*1.05)
plt.grid()
plt.show()
for i in range(len(fractions)):
    print(f"for fraction {fractions[i]}:")
    print(f"Average error: {res_errors[i]}")